In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df_resume = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/Resume/Resume.csv')
df_jd = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/selected_jobs_description.pkl')

print(df_resume.shape)
print(df_jd.shape)

(2484, 4)
(6, 3)


In [11]:
df_resume.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [12]:
df_jd.head()

,category,job_title,job_description
0,INFORMATION-TECHNOLOGY,IT Support Technician Job in Madison,TeamSoft is seeing an IT Support Specialist to...
1,FINANCE,Senior Accountant/Analyst Job in Denver,Would you like to grow your accounting and fin...
2,ENGINEERING,Sr. Process Engineer,Experis is working with a Pharmaceutical start...
3,SALES,Sales Professional Job in Las Vegas,Aflac Insurance Sales Agent While a career in ...
4,HR,Human Resources Manager Job in Dallas,"Human Resource Manager Salary $55,000 - $75,0..."


In [3]:
!python -m spacy download en_core_web_sm
import spacy
nlp = spacy.load('en_core_web_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


It is text preprocessing step in which we are filtering the useful info(like skills) and removing unneccesary noise from our resume.

In [21]:
custom_stopwords = {'include', 'city', 'company', 'work', 'state'}

def clean_text(text):
  doc = nlp(text.lower())
  tokens = [
      token.lemma_
      for token in doc
      if not token.is_stop
      and not token.is_punct
      and not token.is_space
      and token.lemma_.strip() != ''
      and token.lemma_ not in custom_stopwords
  ]

  return ' '.join(tokens)

In [22]:
sample_resume = df_resume[df_resume['Category'] == 'INFORMATION-TECHNOLOGY']['Resume_str'].iloc[0]
cleaned = clean_text(sample_resume)

words = ['state', 'city', 'name', 'company']
if any(word in cleaned[:1000] for word in words):
  print("found")

In [23]:
from collections import Counter
categories = ['INFORMATION-TECHNOLOGY', 'ENGINEERING', 'FINANCE', 'SALES', 'HR', 'HEALTHCARE']

import time
start = time.time()

categories_top_words = {}
for cat in categories:
  texts = df_resume[df_resume['Category'] == cat]['Resume_str'].apply(clean_text)
  all_words = ' '.join(texts).split()
  categories_top_words[cat] = set([word for word, count in Counter(all_words).most_common(30)])

print(f"It took {time.time() - start:.1f} seconds")


It took 143.2 seconds


In [24]:
common_across_all = set.intersection(*categories_top_words.values())
print(common_across_all)

{'customer', 'provide', 'service', 'management'}


Updated the Preprocessing step after knowing there are some words that are unnecessary and carring negligible or irrelevant information.

In [25]:
sample_resume = df_resume[df_resume['Category'] == 'INFORMATION-TECHNOLOGY']['Resume_str'].iloc[0]
cleaned = clean_text(sample_resume)
print(cleaned[:500])

information technology summary dedicated information assurance professional verse analyze mitigate risk find cost effective solution excel boost performance productivity establish realistic goal enforce deadline versatile professional 37 year enterprise design engineering methodology skill enterprise platform knowledge product lifecycle management plm project track hardware software upgrade planning product requirement documentation self direct ms visio decisive collaborative domain active direc


In [27]:
selected_categories = ['INFORMATION-TECHNOLOGY', 'ENGINEERING', 'FINANCE', 'HR', 'SALES', 'HEALTHCARE']
df_selected = df_resume[df_resume['Category'].isin(selected_categories)].copy()
print(df_selected.shape)

(697, 4)


In [29]:
import time
start = time.time()
df_selected['cleaned_resume'] = df_selected['Resume_str'].apply(clean_text)
print(f"took {time.time() - start:.1f} seconds")

took 154.3 seconds


In [31]:
df_jd['cleaned_description'] = df_jd['job_description'].apply(clean_text)
df_jd

,category,job_title,job_description,cleaned_description
0,INFORMATION-TECHNOLOGY,IT Support Technician Job in Madison,TeamSoft is seeing an IT Support Specialist to...,teamsoft see support specialist join client ma...
1,FINANCE,Senior Accountant/Analyst Job in Denver,Would you like to grow your accounting and fin...,like grow accounting finance career great star...
2,ENGINEERING,Sr. Process Engineer,Experis is working with a Pharmaceutical start...,experis pharmaceutical start direct hire sr pr...
3,SALES,Sales Professional Job in Las Vegas,Aflac Insurance Sales Agent While a career in ...,aflac insurance sale agent career sale success...
4,HR,Human Resources Manager Job in Dallas,"Human Resource Manager Salary $55,000 - $75,0...","human resource manager salary $ 55,000 $ 75,00..."
5,HEALTHCARE,Registered Nurse - Clinic Job in Houston,Job Description Position Summary: Provides pro...,job description position summary provide profe...


In [32]:
df_selected.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/resume_cleaned.pkl')
df_jd.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/jd_cleaned.pkl')
print('saved')

saved
